# 🏭 Semana 6 - Dia 2: NN na Prática — BelgoEstoque (Demanda)

Notebook prático para acompanhar [docs/24-dia2-semana6-nn-belgoestoque.md](../../docs/24-dia2-semana6-nn-belgoestoque.md).

**Pergunta-objetivo:** *vale a pena uma rede neural para prever demanda no BelgoEstoque, ou XGBoost resolve melhor?*

**Plano:**
1. Gerar dataset sintético realista de demanda (sazonalidade + tendência + ruído + eventos).
2. Engenharia de features temporais (lags, médias móveis, ciclos).
3. Split temporal correto (sem shuffle).
4. Treinar XGBoost (baseline) e NN (Keras) com mesmas features.
5. Comparar MAE, RMSE, MAPE, bias, tempo e interpretabilidade.
6. Decidir, com base na matriz do dia, qual modelo levar para a Fase 4.

## 1. Setup

In [ ]:
import os, time, warnings
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Imports OK')

## 2. Dataset sintético — demanda semanal por SKU

Vamos simular 3 SKUs do catálogo BelgoEstoque (telas e arames), com 3 anos de histórico semanal:

- **Tendência:** crescimento ou decréscimo lento
- **Sazonalidade anual:** pico em determinadas semanas (construção civil)
- **Sazonalidade mensal:** efeito início/fim de mês
- **Eventos promocionais:** picos pontuais (~5% das semanas)
- **Ruído gaussiano:** comportamento real nunca é perfeito

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

skus = {
    'TELA-1500-50M':  {'base': 120, 'trend': 0.08,  'amp_year': 35, 'amp_month': 8,  'noise': 12},
    'ARAME-FARPADO':  {'base':  80, 'trend': 0.02,  'amp_year': 18, 'amp_month': 5,  'noise':  9},
    'TELA-ALAMBRADO': {'base':  55, 'trend': -0.04, 'amp_year': 12, 'amp_month': 4,  'noise':  7},
}

n_weeks = 156  # 3 anos
datas = pd.date_range('2023-05-01', periods=n_weeks, freq='W-MON')

rows = []
for sku, p in skus.items():
    promo = rng.random(n_weeks) < 0.05  # 5% das semanas tem promo
    feriado = rng.random(n_weeks) < 0.08
    for i, d in enumerate(datas):
        semana_ano = d.isocalendar().week
        mes = d.month
        sazon_ano = p['amp_year'] * np.sin(2 * np.pi * semana_ano / 52)
        sazon_mes = p['amp_month'] * np.sin(2 * np.pi * mes / 12 + 1.0)
        tendencia = p['trend'] * i
        ruido = rng.normal(0, p['noise'])
        boost_promo = 25 if promo[i] else 0
        ajuste_feriado = -8 if feriado[i] else 0
        demanda = max(0, p['base'] + tendencia + sazon_ano + sazon_mes + boost_promo + ajuste_feriado + ruido)
        rows.append({
            'data': d, 'sku': sku, 'semana_ano': semana_ano, 'mes': mes,
            'evento_promocional': int(promo[i]),
            'feriado_proximo': int(feriado[i]),
            'demanda': round(demanda)
        })

df = pd.DataFrame(rows).sort_values(['sku', 'data']).reset_index(drop=True)
print(f'Dataset gerado: {df.shape}')
print(f'Periodo: {df.data.min().date()} -> {df.data.max().date()}')
print(f'SKUs: {df.sku.unique().tolist()}')
df.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, (sku, sub) in zip(axes, df.groupby('sku')):
    ax.plot(sub['data'], sub['demanda'], color='steelblue', linewidth=1.2)
    promos = sub[sub.evento_promocional == 1]
    ax.scatter(promos['data'], promos['demanda'], color='crimson', s=20, label='promo', zorder=5)
    ax.set_title(f'{sku}')
    ax.set_ylabel('demanda')
    ax.legend(loc='upper left')
axes[-1].set_xlabel('data')
plt.tight_layout(); plt.show()

## 3. Feature engineering temporal

Construímos **lags** (vendas passadas), **médias móveis** e features cíclicas (sin/cos) para que ambos modelos tenham o mesmo material para aprender.

In [ ]:
def add_features(group):
    g = group.sort_values('data').copy()
    for lag in [1, 2, 4, 8, 12]:
        g[f'lag_{lag}'] = g['demanda'].shift(lag)
    for w in [4, 8, 12]:
        g[f'mm_{w}'] = g['demanda'].shift(1).rolling(w).mean()
    g['tendencia_4s'] = g['demanda'].shift(1).rolling(4).mean() - g['demanda'].shift(5).rolling(4).mean()
    g['sin_semana'] = np.sin(2 * np.pi * g['semana_ano'] / 52)
    g['cos_semana'] = np.cos(2 * np.pi * g['semana_ano'] / 52)
    g['sin_mes']    = np.sin(2 * np.pi * g['mes'] / 12)
    g['cos_mes']    = np.cos(2 * np.pi * g['mes'] / 12)
    return g

df_feat = df.groupby('sku', group_keys=False).apply(add_features)
df_feat = pd.get_dummies(df_feat, columns=['sku'], prefix='sku', drop_first=False)
df_feat = df_feat.dropna().reset_index(drop=True)

print(f'Apos feature eng: {df_feat.shape}')
df_feat.head()

## 4. Split temporal

**NUNCA** usar `train_test_split(shuffle=True)` aqui. Cortamos por data: tudo até `2025-12-31` é treino, 2026 é teste.

In [ ]:
cutoff = pd.Timestamp('2026-01-01')
feature_cols = [c for c in df_feat.columns if c not in ['data', 'demanda']]

train = df_feat[df_feat.data <  cutoff]
test  = df_feat[df_feat.data >= cutoff]

X_train, y_train = train[feature_cols].astype(float), train['demanda'].astype(float)
X_test,  y_test  = test[feature_cols].astype(float),  test['demanda'].astype(float)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Treino: {train.data.min().date()} -> {train.data.max().date()}')
print(f'Teste:  {test.data.min().date()} -> {test.data.max().date()}')

## 5. Métricas auxiliares

In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, 1, denom)
    return float(np.mean(np.abs(y_pred - y_true) / denom) * 100)

def evaluate(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    sm   = smape(y_true, y_pred)
    return {'modelo': label, 'MAE': mae, 'RMSE': rmse, 'sMAPE_%': sm, 'bias': bias}

## 6. Baseline — XGBoost

Defaults razoáveis, sem tuning. É a régua que a NN precisa bater.

In [ ]:
from xgboost import XGBRegressor

t0 = time.time()
xgb = XGBRegressor(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
)
xgb.fit(X_train, y_train)
tempo_xgb = time.time() - t0

pred_xgb = xgb.predict(X_test)
res_xgb = evaluate(y_test, pred_xgb, 'XGBoost')
res_xgb['tempo_s'] = round(tempo_xgb, 2)
res_xgb

## 7. Rede Neural — Keras

Mesmas features, mas a NN precisa de **scaling**.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(RANDOM_STATE)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

def build_nn(n_features):
    m = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1)  # regressao
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mae', metrics=['mae'])
    return m

nn = build_nn(X_train_s.shape[1])
early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)

t0 = time.time()
history = nn.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=200, batch_size=32,
    callbacks=[early, reduce_lr], verbose=0
)
tempo_nn = time.time() - t0
print(f'NN treinou em {len(history.history["loss"])} epocas | {tempo_nn:.1f}s')

In [ ]:
pred_nn = nn.predict(X_test_s, verbose=0).ravel()
res_nn = evaluate(y_test, pred_nn, 'NN_Keras')
res_nn['tempo_s'] = round(tempo_nn, 2)
res_nn

## 8. Comparação direta

In [ ]:
compare = pd.DataFrame([res_xgb, res_nn]).set_index('modelo').round(3)
compare

In [ ]:
h = history.history
plt.figure(figsize=(10, 4))
plt.plot(h['loss'], label='train')
plt.plot(h['val_loss'], label='val')
plt.title('NN - curva de loss (MAE)')
plt.xlabel('epoca'); plt.ylabel('MAE'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
test_plot = test.copy().reset_index(drop=True)
test_plot['pred_xgb'] = pred_xgb
test_plot['pred_nn']  = pred_nn

skus_uniq = [c.replace('sku_', '') for c in feature_cols if c.startswith('sku_')]
fig, axes = plt.subplots(len(skus_uniq), 1, figsize=(12, 8), sharex=False)
if len(skus_uniq) == 1:
    axes = [axes]

for ax, sku in zip(axes, skus_uniq):
    col = f'sku_{sku}'
    sub = test_plot[test_plot[col] == 1]
    ax.plot(sub['data'], sub['demanda'], label='real', color='black', linewidth=1.5)
    ax.plot(sub['data'], sub['pred_xgb'], label='XGBoost', color='steelblue', linestyle='--')
    ax.plot(sub['data'], sub['pred_nn'],  label='NN',     color='crimson',   linestyle='--')
    ax.set_title(sku)
    ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

## 9. Distribuição do erro

Não basta olhar a média. A cauda do erro é o que vira ruptura ou capital parado em produção.

In [ ]:
err_xgb = pred_xgb - y_test.values
err_nn  = pred_nn  - y_test.values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(err_xgb, bins=30, alpha=0.8, color='steelblue')
axes[0].axvline(0, color='black', linestyle='--')
axes[0].set_title('XGBoost - erro (pred - real)')
axes[1].hist(err_nn, bins=30, alpha=0.8, color='crimson')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('NN - erro (pred - real)')
plt.tight_layout(); plt.show()

print('Estatisticas do erro absoluto:')
for nome, e in [('XGBoost', err_xgb), ('NN', err_nn)]:
    abs_e = np.abs(e)
    print(f'  {nome}: mediana {np.median(abs_e):.1f} | p90 {np.quantile(abs_e, 0.9):.1f} | max {abs_e.max():.1f}')

## 10. Interpretabilidade — só XGBoost entrega de graça

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=True).tail(12)
plt.figure(figsize=(8, 5))
imp.plot(kind='barh', color='steelblue')
plt.title('XGBoost - top 12 features')
plt.xlabel('importance'); plt.tight_layout(); plt.show()

## 11. Matriz de decisão BelgoEstoque

Aplicando os critérios do [doc do dia](../../docs/24-dia2-semana6-nn-belgoestoque.md#7-critérios-de-decisão-belgoestoque):

In [ ]:
ganho_pct = 100 * (res_xgb['MAE'] - res_nn['MAE']) / res_xgb['MAE']
print(f'Ganho da NN sobre XGBoost (MAE): {ganho_pct:+.2f}%')
print(f'Tempo de treino: XGBoost {res_xgb["tempo_s"]}s vs NN {res_nn["tempo_s"]}s')
print()

criterios = pd.DataFrame([
    {'criterio': 'NN >5% melhor em MAE?',                        'resposta': 'sim' if ganho_pct > 5 else 'nao'},
    {'criterio': 'Diferenca estavel em todos SKUs?',             'resposta': 'avaliar visualmente'},
    {'criterio': 'Time tem rotina de retreino e monitoramento?', 'resposta': 'nao (Belgo ainda nao)'},
    {'criterio': 'Cliente aceita caixa-preta?',                  'resposta': 'nao (operacao quer interpretar)'},
    {'criterio': 'Ha GPU disponivel?',                           'resposta': 'nao'},
])
criterios

## 12. Conclusão

Com 3 SKUs, 156 semanas, features de lag/sazonalidade e split temporal limpo:

- **XGBoost** entrega previsão decente em segundos, com feature importance pronta para auditar.
- **NN** atinge MAE comparável, mas exige scaling, mais épocas, e não fornece interpretabilidade direta.
- Em 4 dos 5 critérios da matriz, **XGBoost vence** para o estado atual da Belgo.

**Decisão para a Fase 4 do BelgoEstoque:**

> Começar com **XGBoost** em produção. Reavaliar NN somente quando: (a) catálogo crescer para centenas de SKUs com correlação, (b) a previsão precisar ser multi-passo (4+ semanas correlacionadas), ou (c) houver budget para infraestrutura e monitoramento de DL.

**Não é derrota — é boa engenharia.**

## 13. Exercícios sugeridos

1. Triplique o ruído (`noise`) no dataset e refaça a comparação. Quem degrada menos?
2. Adicione 50 SKUs sintéticos (mesma estrutura, parâmetros aleatórios). A NN começa a vencer?
3. Faça previsão multi-step (4 semanas à frente) — XGBoost precisa de loop, NN pode ter saída de 4 neurônios. Compare a complexidade.
4. Substitua o split por `TimeSeriesSplit(n_splits=5)` e calcule média ± desvio do MAE.